In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import convolve

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_IMAGES_DIR = PROCESSED_DIR / "processed_images"

GRAY_DIR = PROCESSED_IMAGES_DIR / "gray"
BINARY_DIR = PROCESSED_IMAGES_DIR / "binary"
SKELETON_DIR = PROCESSED_IMAGES_DIR / "skeleton"

image_mapping = pd.read_csv(PROCESSED_DIR / "image_mapping.csv")

print(image_mapping.shape)
image_mapping.head()

(89, 3)


,image_name,image_path,case_id
0,١_page-0001.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,1
1,١_page-0002.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,2
2,١_page-0003.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,3
3,١_page-0004.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,4
4,١_page-0005.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,5


In [ ]:
def count_skeleton_points(skeleton_img):
    """
    Counts endpoints and intersections in a skeleton image.
    Endpoint = skeleton pixel with 1 neighbor.
    Intersection = skeleton pixel with 3 or more neighbors.
    """
    
    ## skeleton make the drwaing lines to be one pixel wide(very thin) to make it easier to analyze the structure of the
    # drawing and extract features like endpoints and intersections.
    ## this line to make sure that the image is binary and in uint8(unsigned integer) format (0 and 1)
    skel = (skeleton_img > 0).astype(np.uint8)  

    kernel = np.array([ ## the mid pixel is zero bc we wanna only count neighbors not the pixel itself
        [1, 1, 1],
        [1, 0, 1],
        [1, 1, 1]
    ], dtype=np.uint8)

    neighbor_count = convolve(skel, kernel, mode="constant", cval=0) ## cval mean like zero padding around image
## endpoint is a pixel in the skeleton that has only one neighbor, which indicates the end of a line segment.
## intersection is a pixel in the skeleton that has three or more neighbors, which indicates a junction
    endpoints = np.logical_and(skel == 1, neighbor_count == 1).sum()
    intersections = np.logical_and(skel == 1, neighbor_count >= 3).sum()

    return int(endpoints), int(intersections)

In [ ]:
def extract_image_features(case_id):
    gray_path = GRAY_DIR / f"case_{case_id:03d}_gray.png"
    binary_path = BINARY_DIR / f"case_{case_id:03d}_binary.png"
    skeleton_path = SKELETON_DIR / f"case_{case_id:03d}_skeleton.png"

    gray = cv2.imread(str(gray_path), cv2.IMREAD_GRAYSCALE)
    binary = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    skeleton = cv2.imread(str(skeleton_path), cv2.IMREAD_GRAYSCALE)

    if gray is None or binary is None or skeleton is None:
        raise ValueError(f"Missing processed image for case {case_id}")

    h, w = binary.shape
    image_area = h * w

    # Binary foreground pixels
    ink_pixels = int(np.count_nonzero(binary)) ## // count nozero pixels in the binary image which represent the drawn lines
    ink_density = ink_pixels / image_area  ##ratio of ink pixels to total pixels(how much drawing??)

    # Skeleton pixels
    skeleton_pixels = int(np.count_nonzero(skeleton)) ## the length of image skeleton which is a proxy for the total length of drawn lines
    skeleton_density = skeleton_pixels / image_area ## the intensity of drwaing line after making it one pixel wide

    # Stroke thickness proxy
    stroke_thickness_proxy = ink_pixels / (skeleton_pixels + 1e-6) ## the thickness of drawn lines 

    # Heavy line score: نفس الفكرة كمؤشر بسيط لـ repeated/heavy strokes
    heavy_line_score = stroke_thickness_proxy

    ## any conncted region of foreground pixels in binary image is considred as a component
    ## num_labels = number of connected components (including background as label 0 -> state[0] is background)
    ## labels = label image where each pixel is labeled with its component id
    ## stats = statistics for each component (bounding box, area,position of the component ...)
    ## centroids = centroid of each component
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    # label 0 = background, so we ignore it
    component_areas = stats[1:, cv2.CC_STAT_AREA]

    valid_components = component_areas[component_areas > 20]
    connected_components_count = int(len(valid_components))

    # Contours
    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL, ## RETR_EXTERNAL -> only the outer contours
        cv2.CHAIN_APPROX_SIMPLE
    )

    valid_contours = [
        cnt for cnt in contours
        if cv2.contourArea(cnt) > 20
    ]

    contour_count = len(valid_contours)

    ## the higher number of endpoints indicates -> more details, more disconnected lines, complete shapes
    ## the higher number of intersections indicates -> more complex shapes, overlapping lines
    endpoints_count, intersections_count = count_skeleton_points(skeleton)

    # Bounding box around all drawing pixels
    ys, xs = np.where(binary > 0) ## check first for nonzero pixels in binary image to get their coordinates (x and y)

    if len(xs) > 0 and len(ys) > 0:
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        bbox_width = int(x_max - x_min + 1)
        bbox_height = int(y_max - y_min + 1)
        bbox_area = bbox_width * bbox_height
        bbox_area_ratio = bbox_area / image_area
    else:
        bbox_width = 0
        bbox_height = 0
        bbox_area_ratio = 0

    # Orientation type without rotating
    is_landscape = int(w > h) ## if width > height -> landscape (1) else portrait (0)

    features = {
        "case_id": case_id,
        "image_width": w,
        "image_height": h,
        "is_landscape": is_landscape,
        "ink_pixels": ink_pixels,
        "ink_density": ink_density,
        "skeleton_length": skeleton_pixels,
        "skeleton_density": skeleton_density,
        "stroke_thickness_proxy": stroke_thickness_proxy,
        "heavy_line_score": heavy_line_score,
        "connected_components_count": connected_components_count,
        "contour_count": contour_count,
        "endpoints_count": endpoints_count,
        "intersections_count": intersections_count,
        "bbox_width": bbox_width,
        "bbox_height": bbox_height,
        "bbox_area_ratio": bbox_area_ratio
    }

    return features

In [5]:
features = extract_image_features(case_id=1)

features

{'case_id': 1,
 'image_width': 1241,
 'image_height': 1755,
 'is_landscape': 0,
 'ink_pixels': 27946,
 'ink_density': 0.012831302758780598,
 'skeleton_length': 7015,
 'skeleton_density': 0.0032209113595092643,
 'stroke_thickness_proxy': 3.983749108484141,
 'heavy_line_score': 3.983749108484141,
 'connected_components_count': 132,
 'contour_count': 106,
 'endpoints_count': 376,
 'intersections_count': 59,
 'bbox_width': 1200,
 'bbox_height': 1687,
 'bbox_area_ratio': 0.9294957884804782}

In [6]:
all_features = []

for _, row in image_mapping.iterrows():
    case_id = int(row["case_id"])
    features = extract_image_features(case_id)
    all_features.append(features)

image_features = pd.DataFrame(all_features)

print(image_features.shape)
image_features.head()

(89, 17)


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,skeleton_length,skeleton_density,stroke_thickness_proxy,heavy_line_score,connected_components_count,contour_count,endpoints_count,intersections_count,bbox_width,bbox_height,bbox_area_ratio
0,1,1241,1755,0,27946,0.012831,7015,0.003221,3.983749,3.983749,132,106,376,59,1200,1687,0.929496
1,2,1241,1755,0,33761,0.015501,6718,0.003085,5.025454,5.025454,142,114,428,362,1183,1643,0.892428
2,3,1241,1755,0,29999,0.013774,7692,0.003532,3.900026,3.900026,122,104,416,245,1165,1494,0.799149
3,4,1755,1241,1,31808,0.014605,7157,0.003286,4.444320,4.444320,142,111,486,182,1512,1155,0.801835
4,5,1241,1755,0,33813,0.015525,7413,0.003404,4.561311,4.561311,138,116,401,157,1191,1571,0.859091


In [7]:
output_path = PROCESSED_DIR / "image_features.csv"

image_features.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)
image_features.head()

Saved to: d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\data\processed\image_features.csv


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,skeleton_length,skeleton_density,stroke_thickness_proxy,heavy_line_score,connected_components_count,contour_count,endpoints_count,intersections_count,bbox_width,bbox_height,bbox_area_ratio
0,1,1241,1755,0,27946,0.012831,7015,0.003221,3.983749,3.983749,132,106,376,59,1200,1687,0.929496
1,2,1241,1755,0,33761,0.015501,6718,0.003085,5.025454,5.025454,142,114,428,362,1183,1643,0.892428
2,3,1241,1755,0,29999,0.013774,7692,0.003532,3.900026,3.900026,122,104,416,245,1165,1494,0.799149
3,4,1755,1241,1,31808,0.014605,7157,0.003286,4.444320,4.444320,142,111,486,182,1512,1155,0.801835
4,5,1241,1755,0,33813,0.015525,7413,0.003404,4.561311,4.561311,138,116,401,157,1191,1571,0.859091
